# 🛒 Superstore 電商營運分析
**目標**：透過 EDA 找出銷售與利潤的關鍵驅動因子，提供商業洞察。

## 1. 載入資料

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# 設定中文字體（如有需要）
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/superstore.csv', encoding='latin1')
print(f'資料筆數：{df.shape[0]:,}，欄位數：{df.shape[1]}')
df.head()

## 2. 基本資訊

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. 資料品質檢查

In [ ]:
print('=== 缺失值 ===')
print(df.isnull().sum())
print(f'\n=== 重複筆數：{df.duplicated().sum()} ===')

## 4. 資料前處理

In [ ]:
# 日期轉換
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# 新增衍生欄位
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.to_period('M').astype(str)  # 轉成字串，SQLite 才支援
df['Profit Margin'] = df['Profit'] / df['Sales']  # 利潤率
df['Ship Days'] = (df['Ship Date'] - df['Order Date']).dt.days  # 出貨天數

print('前處理完成！新增欄位：Year, Month, Profit Margin, Ship Days')
df[['Sales', 'Profit', 'Profit Margin', 'Ship Days']].describe()

## 5. SQL 分析（業界常用技能展示）

In [ ]:
# 建立 SQLite in-memory DB
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

# 各品類銷售與利潤彙總
query = '''
SELECT 
    Category,
    COUNT(*) AS order_count,
    ROUND(SUM(Sales), 0) AS total_sales,
    ROUND(SUM(Profit), 0) AS total_profit,
    ROUND(AVG("Profit Margin") * 100, 1) AS avg_margin_pct
FROM orders
GROUP BY Category
ORDER BY total_sales DESC
'''
pd.read_sql(query, conn)

In [ ]:
# 折扣對利潤的影響
query2 = '''
SELECT 
    CASE 
        WHEN Discount = 0 THEN '無折扣'
        WHEN Discount <= 0.2 THEN '低折扣 (<=20%)'
        WHEN Discount <= 0.4 THEN '中折扣 (21-40%)'
        ELSE '高折扣 (>40%)'
    END AS discount_group,
    COUNT(*) AS order_count,
    ROUND(AVG("Profit Margin") * 100, 1) AS avg_margin_pct
FROM orders
GROUP BY discount_group
ORDER BY avg_margin_pct DESC
'''
pd.read_sql(query2, conn)

## 6. 視覺化分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：各品類銷售 vs 利潤
cat_summary = df.groupby('Category')[['Sales', 'Profit']].sum().reset_index()
x = range(len(cat_summary))
axes[0].bar([i - 0.2 for i in x], cat_summary['Sales'], width=0.4, label='Sales', color='steelblue')
axes[0].bar([i + 0.2 for i in x], cat_summary['Profit'], width=0.4, label='Profit', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cat_summary['Category'])
axes[0].set_title('各品類：銷售額 vs 利潤')
axes[0].legend()

# 右：折扣 vs 利潤率散佈圖
axes[1].scatter(df['Discount'], df['Profit Margin'], alpha=0.3, color='mediumseagreen')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Discount')
axes[1].set_ylabel('Profit Margin')
axes[1].set_title('折扣率 vs 利潤率（紅線=損益兩平）')

plt.tight_layout()
plt.savefig('../outputs/category_analysis.png', dpi=150)
plt.show()

In [ ]:
# 月度銷售趨勢
monthly = df.groupby('Month')[['Sales', 'Profit']].sum().reset_index()
monthly['Month'] = monthly['Month'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly['Month'], monthly['Sales'], color='steelblue', alpha=0.6, label='Sales')
ax2.plot(monthly['Month'], monthly['Profit'], color='coral', marker='o', linewidth=2, label='Profit')
ax1.set_xlabel('Month')
ax1.set_ylabel('Sales', color='steelblue')
ax2.set_ylabel('Profit', color='coral')
plt.xticks(rotation=45)
ax1.set_title('月度銷售額 & 利潤趨勢')
plt.tight_layout()
plt.savefig('../outputs/monthly_trend.png', dpi=150)
plt.show()

## 7. 商業洞察總結

（跑完上面的分析後，在這裡寫你的結論，例如：）

- 📌 **折扣陷阱**：折扣超過 X% 的訂單平均利潤率為負，建議檢討折扣策略
- 📌 **品類差異**：Technology 銷售額最高但利潤率不一定最好
- 📌 **季節性**：Q4（11-12月）銷售明顯拉升，建議提前備貨